###Installing Required Packages

In [0]:
%pip install openmeteo-requests pvlib pandas pyarrow xgboost scikit-learn mlflow
dbutils.library.restartPython()

### Testing API call 
with 1-Month of historic data

In [0]:
# import openmeteo_requests
# import pandas as pd

# # 1. Fetch historical data (Perth, WA)
# om = openmeteo_requests.Client()
# params = {
# 	"latitude": -31.95,
# 	"longitude": 115.86,
# 	"start_date": "2024-01-01",
# 	"end_date": "2024-01-31",
# 	"hourly": ["direct_normal_irradiance", "diffuse_radiation"]
# }
# response = om.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)[0]
# hourly = response.Hourly()

# # 2. Format into a Pandas DataFrame
# date_range = pd.date_range(
#     start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
#     end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
#     freq=pd.Timedelta(seconds=hourly.Interval()),
#     inclusive="left"
# )

# pdf_raw = pd.DataFrame({
#     "timestamp": date_range,
#     "dni": hourly.Variables(0).ValuesAsNumpy(),
#     "dhi": hourly.Variables(1).ValuesAsNumpy()
# })

# # 3. Convert to a Distributed Spark DataFrame (Bronze Layer)
# df_bronze = spark.createDataFrame(pdf_raw)
# df_bronze.display()

### Testing feature engineering
applied geometric effect correction

In [0]:
# import pvlib
# import numpy as np
# from pyspark.sql.functions import pandas_udf
# import pyspark.sql.types as T
# import pandas as pd

# # Define panel geometry
# TILT = 32.0 
# AZIMUTH = 0.0 # Facing North
# LAT = -31.95
# LON = 115.86

# @pandas_udf("float")
# def calculate_gti(timestamps: pd.Series, dni: pd.Series, dhi: pd.Series) -> pd.Series:
#     # 1. Calculate Sun Position (Convert to DatetimeIndex for pvlib)
#     solpos = pvlib.solarposition.get_solarposition(pd.DatetimeIndex(timestamps), LAT, LON)
    
#     # Strip the DatetimeIndex to prevent Pandas outer-join alignment errors
#     zenith = solpos['apparent_zenith'].to_numpy()
#     azimuth = solpos['azimuth'].to_numpy()
    
#     # 2. Calculate Global Tilted Irradiance (GTI)
#     poa_irrad = pvlib.irradiance.get_total_irradiance(
#         surface_tilt=TILT,
#         surface_azimuth=AZIMUTH,
#         dni=dni,
#         ghi=dhi + (dni * np.cos(np.radians(zenith))), 
#         dhi=dhi,
#         solar_zenith=zenith,
#         solar_azimuth=azimuth
#     )
    
#     return poa_irrad['poa_global']

# # 3. Apply function
# df_silver = df_bronze.withColumn(
#     "global_tilted_irradiance[W/m^2]", 
#     calculate_gti(df_bronze["timestamp"], df_bronze["dni"], df_bronze["dhi"])
# )

# df_silver.display()

### Data Ingestion
Collect variables over 10-year window with yearly batching to avoid memory buffer overflows

In [0]:
import openmeteo_requests
import pandas as pd

om = openmeteo_requests.Client()
years = list(range(2016, 2026))
all_dfs = []

print("Starting cloud-tier batched data ingestion...")
for year in years:
    print(f"Fetching data for year: {year}...")
    
    params = {
        "latitude": -31.95,
        "longitude": 115.86,
        "start_date": f"{year}-01-01",
        "end_date": f"{year}-12-31",
        "hourly": [
            "diffuse_radiation",
            "cloud_cover_low",   
            "cloud_cover_mid",   
            "cloud_cover_high",  
            "total_column_integrated_water_vapour",
            "sunshine_duration",
            "direct_normal_irradiance",
            "temperature_2m",        
            "relative_humidity_2m",  
            "surface_pressure"       
        ]
    }
    
    response = om.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)[0]
    hourly = response.Hourly()
    
    date_range = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )
    
    pdf_year = pd.DataFrame({
        "timestamp": date_range,
        "dhi": hourly.Variables(0).ValuesAsNumpy(),
        "cloud_low": hourly.Variables(1).ValuesAsNumpy(),   
        "cloud_mid": hourly.Variables(2).ValuesAsNumpy(),   
        "cloud_high": hourly.Variables(3).ValuesAsNumpy(),  
        "water_vapour": hourly.Variables(4).ValuesAsNumpy(),
        "sunshine_duration": hourly.Variables(5).ValuesAsNumpy(),
        "dni": hourly.Variables(6).ValuesAsNumpy(),
        "temperature": hourly.Variables(7).ValuesAsNumpy(),      
        "relative_humidity": hourly.Variables(8).ValuesAsNumpy(),
        "surface_pressure": hourly.Variables(9).ValuesAsNumpy()   
    })
    
    all_dfs.append(pdf_year)

pdf_combined = pd.concat(all_dfs, ignore_index=True)
df_bronze = spark.createDataFrame(pdf_combined)
print(f"Ingestion complete! Total rows loaded: {df_bronze.count()}")


In [0]:
import pvlib
import pandas as pd
import numpy as np
from pyspark.sql.functions import pandas_udf, col, when
import pyspark.sql.types as T

LAT = -31.95
LON = 115.86

# 1. Expand the UDF schema to return Zenith along with theoretical DNI/GHI
cs_schema = T.StructType([
    T.StructField("clear_dni", T.FloatType(), True),
    T.StructField("clear_ghi", T.FloatType(), True),
    T.StructField("zenith", T.FloatType(), True)  
])

@pandas_udf(cs_schema)
def get_clear_sky_and_zenith(timestamps: pd.Series) -> pd.DataFrame:
    tus = pd.DatetimeIndex(timestamps)
    loc = pvlib.location.Location(LAT, LON)
    
    # Get clear sky components
    cs = loc.get_clearsky(tus)
    # Get solar position matrix to extract zenith
    solpos = loc.get_solarposition(tus)
    
    return pd.DataFrame({
        "clear_dni": cs['dni'].astype('float32'),
        "clear_ghi": cs['ghi'].astype('float32'),
        "zenith": solpos['apparent_zenith'].astype('float32')
    })

# 2. Extract components into the Spark DataFrame
df_theoretical = df_bronze.withColumn(
    "cs_components", get_clear_sky_and_zenith(col("timestamp"))
).select(
    "*",
    col("cs_components.clear_dni").alias("cs_dni"),
    col("cs_components.clear_ghi").alias("cs_ghi"),
    col("cs_components.zenith").alias("zenith")
).drop("cs_components")

# Calculate Factors and normalize tiered cloud features
df_silver_factors = df_theoretical.withColumn(
    "direct_clear_sky_factor",
    when((col("cs_dni") > 5.0) & (col("zenith") < 85.0), col("dni") / col("cs_dni")).otherwise(0.0)
).withColumn(
    "diffuse_clear_sky_factor",
    when((col("cs_ghi") > 5.0) & (col("zenith") < 85.0), col("dhi") / col("cs_ghi")).otherwise(0.0)
).withColumn(
    "direct_clear_sky_factor",
    when(col("direct_clear_sky_factor") > 1.0, 1.0).otherwise(col("direct_clear_sky_factor"))
).withColumn(
    "diffuse_clear_sky_factor",
    when(col("diffuse_clear_sky_factor") > 1.0, 1.0).otherwise(col("diffuse_clear_sky_factor"))
).withColumn(
    "sunshine_fraction", col("sunshine_duration") / 3600.0
).withColumn(
    "cloud_low_fraction", col("cloud_low") / 100.0   
).withColumn(
    "cloud_mid_fraction", col("cloud_mid") / 100.0   
).withColumn(
    "cloud_high_fraction", col("cloud_high") / 100.0 
).withColumn(
    "rh_fraction", col("relative_humidity") / 100.0  
).select(
    "timestamp",
    "cloud_low_fraction",
    "cloud_mid_fraction",
    "cloud_high_fraction",
    "water_vapour",
    "sunshine_fraction",
    "temperature",        
    "rh_fraction",         
    "surface_pressure",    
    "direct_clear_sky_factor",
    "diffuse_clear_sky_factor"
)

print("Layered Cloud Silver Layer complete!")
# df_silver_factors.display()


### Chronological Time-Aware Model Benchmark

This final training section replaces the earlier random train/test split workflow for reported metrics. It imports the shared `src/solar_yield` feature contract, compares the static feature set against the time-aware feature set on chronological validation windows, and logs the benchmark plus the time-aware model to MLflow.


In [ ]:
from pathlib import Path
import json
import math
import sys
import tempfile

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import RegressorChain
from xgboost import XGBRegressor


def add_repo_src_to_path():
    candidates = []
    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents])

    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        workspace_path = Path("/Workspace") / notebook_path.lstrip("/")
        workspace_dir = workspace_path.parent
        candidates.extend([workspace_dir, *workspace_dir.parents])
    except Exception:
        pass

    for candidate in candidates:
        src_path = candidate / "src"
        if (src_path / "solar_yield" / "features.py").exists():
            src_path_text = str(src_path)
            if src_path_text not in sys.path:
                sys.path.insert(0, src_path_text)
            return candidate

    raise RuntimeError("Could not locate src/solar_yield/features.py from this notebook.")


repo_root = add_repo_src_to_path()
print(f"Using shared package source from: {repo_root / 'src'}")

from solar_yield.features import (  # noqa: E402
    BASE_FEATURE_COLUMNS,
    FEATURE_COLUMNS,
    TARGET_COLUMNS,
    add_model_features,
    calculate_interior_weights,
    prepare_model_matrix,
)

TRAIN_START = pd.Timestamp("2016-01-01")
VALIDATION_START = pd.Timestamp("2024-01-01")
HOLDOUT_START = pd.Timestamp("2025-01-01")
HOLDOUT_END = pd.Timestamp("2026-01-01")
FEATURE_SET_VERSION = "time_weather_dynamics_v1"

raw_data = df_silver_factors.select(
    "timestamp",
    *BASE_FEATURE_COLUMNS,
    *TARGET_COLUMNS,
).toPandas()

raw_data["timestamp"] = pd.to_datetime(raw_data["timestamp"])
if raw_data["timestamp"].dt.tz is not None:
    raw_data["timestamp"] = raw_data["timestamp"].dt.tz_convert(None)

raw_data = raw_data.sort_values("timestamp").dropna(
    subset=[*BASE_FEATURE_COLUMNS, *TARGET_COLUMNS]
).reset_index(drop=True)


def split_by_period(frame):
    timestamps = frame["timestamp"]
    train = frame[(timestamps >= TRAIN_START) & (timestamps < VALIDATION_START)].copy()
    validation = frame[(timestamps >= VALIDATION_START) & (timestamps < HOLDOUT_START)].copy()
    holdout = frame[(timestamps >= HOLDOUT_START) & (timestamps < HOLDOUT_END)].copy()

    if train.empty or validation.empty or holdout.empty:
        raise ValueError(
            "Chronological split produced an empty partition: "
            f"train={len(train)}, validation={len(validation)}, holdout={len(holdout)}"
        )
    return train, validation, holdout


static_data = raw_data[raw_data["sunshine_fraction"] > 0.0].copy()
time_aware_data = add_model_features(raw_data, mode="training")
time_aware_data = time_aware_data[time_aware_data["sunshine_fraction"] > 0.0].copy()

static_train, static_validation, static_holdout = split_by_period(static_data)
time_train, time_validation, time_holdout = split_by_period(time_aware_data)

print(
    "Static rows by split: "
    f"train={len(static_train)}, validation={len(static_validation)}, holdout={len(static_holdout)}"
)
print(
    "Time-aware rows by split: "
    f"train={len(time_train)}, validation={len(time_validation)}, holdout={len(time_holdout)}"
)


def make_xgboost():
    return XGBRegressor(
        n_estimators=500,
        learning_rate=0.01,
        max_depth=5,
        subsample=0.6,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        objective="reg:logistic",
    )


def fit_weighted_chain(train_frame, feature_columns):
    x_train = prepare_model_matrix(train_frame, feature_columns)
    y_train = train_frame[TARGET_COLUMNS]
    sample_weights = calculate_interior_weights(y_train)

    model = RegressorChain(make_xgboost(), order=[0, 1])
    model.fit(x_train, y_train, sample_weight=sample_weights)
    return model


def summarize_predictions(model_name, split_name, y_true, predictions):
    direct_mask = y_true[TARGET_COLUMNS[0]].between(0.01, 0.99, inclusive="neither")
    diffuse_mask = y_true[TARGET_COLUMNS[1]].between(0.01, 0.99, inclusive="neither")

    direct_r2 = np.nan
    diffuse_r2 = np.nan
    if direct_mask.sum() >= 2:
        direct_r2 = r2_score(y_true.loc[direct_mask, TARGET_COLUMNS[0]], predictions[direct_mask, 0])
    if diffuse_mask.sum() >= 2:
        diffuse_r2 = r2_score(
            y_true.loc[diffuse_mask, TARGET_COLUMNS[1]],
            predictions[diffuse_mask, 1],
        )

    return {
        "model_name": model_name,
        "split": split_name,
        "rows": len(y_true),
        "direct_rmse": math.sqrt(mean_squared_error(y_true.iloc[:, 0], predictions[:, 0])),
        "diffuse_rmse": math.sqrt(mean_squared_error(y_true.iloc[:, 1], predictions[:, 1])),
        "direct_mae": mean_absolute_error(y_true.iloc[:, 0], predictions[:, 0]),
        "diffuse_mae": mean_absolute_error(y_true.iloc[:, 1], predictions[:, 1]),
        "direct_interior_r2": direct_r2,
        "diffuse_interior_r2": diffuse_r2,
    }


def evaluate_model(model_name, model, feature_columns, frames_by_split):
    rows = []
    for split_name, frame in frames_by_split.items():
        x_eval = prepare_model_matrix(frame, feature_columns)
        y_eval = frame[TARGET_COLUMNS]
        predictions = model.predict(x_eval)
        rows.append(summarize_predictions(model_name, split_name, y_eval, predictions))
    return rows


def persistence_24h_predictions(eval_frame, history_frame):
    history = history_frame[["timestamp", *TARGET_COLUMNS]].drop_duplicates("timestamp")
    history = history.set_index("timestamp").sort_index()
    previous_timestamps = eval_frame["timestamp"] - pd.Timedelta(hours=24)
    predictions = history.reindex(previous_timestamps)[TARGET_COLUMNS]

    fallback_history = history[history.index < eval_frame["timestamp"].min()]
    if fallback_history.empty:
        fallback_history = history
    fallback_values = fallback_history[TARGET_COLUMNS].median()
    return predictions.fillna(fallback_values).to_numpy()


def evaluate_persistence_baseline(full_frame, frames_by_split):
    rows = []
    for split_name, frame in frames_by_split.items():
        predictions = persistence_24h_predictions(frame, full_frame)
        rows.append(
            summarize_predictions(
                "target_persistence_24h",
                split_name,
                frame[TARGET_COLUMNS],
                predictions,
            )
        )
    return rows


def feature_importance_frame(model, feature_columns):
    rows = []
    order = list(model.order_)
    for step, target_index in enumerate(order):
        estimator = model.estimators_[step]
        importances = getattr(estimator, "feature_importances_", None)
        if importances is None:
            continue

        chain_features = list(feature_columns) + [
            f"chain_{TARGET_COLUMNS[index]}" for index in order[:step]
        ]
        if len(chain_features) != len(importances):
            chain_features = [f"feature_{i}" for i in range(len(importances))]

        rows.extend(
            {
                "target": TARGET_COLUMNS[target_index],
                "feature": feature,
                "importance": float(importance),
            }
            for feature, importance in zip(chain_features, importances)
        )
    return pd.DataFrame(rows)


static_model = fit_weighted_chain(static_train, BASE_FEATURE_COLUMNS)
time_aware_model = fit_weighted_chain(time_train, FEATURE_COLUMNS)

benchmark_rows = []
benchmark_rows.extend(
    evaluate_persistence_baseline(
        static_data,
        {"validation_2024": static_validation, "holdout_2025": static_holdout},
    )
)
benchmark_rows.extend(
    evaluate_model(
        "static_weighted_chain_xgboost",
        static_model,
        BASE_FEATURE_COLUMNS,
        {"validation_2024": static_validation, "holdout_2025": static_holdout},
    )
)
benchmark_rows.extend(
    evaluate_model(
        "time_aware_weighted_chain_xgboost",
        time_aware_model,
        FEATURE_COLUMNS,
        {"validation_2024": time_validation, "holdout_2025": time_holdout},
    )
)

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df["total_rmse"] = benchmark_df["direct_rmse"] + benchmark_df["diffuse_rmse"]
validation_scores = benchmark_df[benchmark_df["split"] == "validation_2024"]
best_validation_row = validation_scores.sort_values("total_rmse").iloc[0]

print("Chronological benchmark results:")
display(benchmark_df.sort_values(["split", "total_rmse"]))
print(
    "Best validation model: "
    f"{best_validation_row['model_name']} "
    f"with total RMSE={best_validation_row['total_rmse']:.4f}"
)

with mlflow.start_run(run_name="Chronological_Time_Aware_Solar_Model") as run:
    mlflow.log_param("feature_set_version", FEATURE_SET_VERSION)
    mlflow.log_param("train_period", "2016-01-01_to_2023-12-31")
    mlflow.log_param("validation_period", "2024-01-01_to_2024-12-31")
    mlflow.log_param("holdout_period", "2025-01-01_to_2025-12-31")
    mlflow.log_param("base_feature_count", len(BASE_FEATURE_COLUMNS))
    mlflow.log_param("time_aware_feature_count", len(FEATURE_COLUMNS))
    mlflow.log_param("selected_by_validation", best_validation_row["model_name"])
    mlflow.log_param("model_family", "RegressorChain_XGBRegressor")

    for _, row in benchmark_df.iterrows():
        metric_prefix = f"{row['model_name']}_{row['split']}"
        for metric_name in [
            "direct_rmse",
            "diffuse_rmse",
            "direct_mae",
            "diffuse_mae",
            "direct_interior_r2",
            "diffuse_interior_r2",
            "total_rmse",
        ]:
            metric_value = row[metric_name]
            if pd.notna(metric_value):
                mlflow.log_metric(f"{metric_prefix}_{metric_name}", float(metric_value))

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        benchmark_path = tmp_path / "chronological_benchmark.csv"
        feature_columns_path = tmp_path / "time_aware_feature_columns.json"
        static_importance_path = tmp_path / "static_feature_importance.csv"
        time_importance_path = tmp_path / "time_aware_feature_importance.csv"

        benchmark_df.to_csv(benchmark_path, index=False)
        feature_columns_path.write_text(json.dumps(FEATURE_COLUMNS, indent=2))
        feature_importance_frame(static_model, BASE_FEATURE_COLUMNS).to_csv(
            static_importance_path,
            index=False,
        )
        feature_importance_frame(time_aware_model, FEATURE_COLUMNS).to_csv(
            time_importance_path,
            index=False,
        )

        mlflow.log_artifact(str(benchmark_path))
        mlflow.log_artifact(str(feature_columns_path))
        mlflow.log_artifact(str(static_importance_path))
        mlflow.log_artifact(str(time_importance_path))

    time_model_info = mlflow.sklearn.log_model(
        sk_model=time_aware_model,
        name="solar_factor_model",
        input_example=prepare_model_matrix(time_train, FEATURE_COLUMNS).head(3),
    )
    static_model_info = mlflow.sklearn.log_model(
        sk_model=static_model,
        name="static_solar_factor_model",
        input_example=prepare_model_matrix(static_train, BASE_FEATURE_COLUMNS).head(3),
    )
    mlflow.log_param("time_aware_model_uri", time_model_info.model_uri)
    mlflow.log_param("static_model_uri", static_model_info.model_uri)

    print("=" * 60)
    print("CHRONOLOGICAL TIME-AWARE MODEL RUN LOGGED")
    print(f"MLflow Run ID for fallback loading: {run.info.run_id}")
    print(f"Use model_uri for inference: {time_model_info.model_uri}")
    print("Fallback run URI: runs:/{run.info.run_id}/solar_factor_model")
    print("=" * 60)
